# SRM Patient Report


### Select Prep LIMS ID


In [ ]:
prep_lims_id = "05T74"

### Training Mode

Training mode uses dev POCKet credentials to prevent changes to production DB.
**Requires `~/work/.dev-env` is setup by SFE.** Otherwise standard `~/work/.prod-env` is used.
Can be run on standard https://jhub.prod.procan.org.au/ in any case.


In [ ]:
training_mode = False

Add Skyline Source File (Leave empty to use POCKet Data) -- empty means `skyline_input_path = ""`


In [ ]:
skyline_input_path = ""

In [ ]:
import logging
import os
import warnings
from datetime import datetime

import dotenv
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import pandas as pd
import procanswon
import seaborn as sns
from textwrap import wrap
import trino
from matplotlib.colors import TwoSlopeNorm
from pandas import DataFrame
from scipy.stats import norm
from sqlalchemy.engine import create_engine
from sqlalchemy import text

In [ ]:
today = datetime.now().strftime("%Y-%m-%d")

In [ ]:
sns.set_theme(color_codes=True)
# Suppress warnings like:
# /Users/olivson/opt/miniconda3/envs/srm-patient-report/lib/python3.13/site-packages/seaborn/categorical.py:3399: UserWarning: 24.7% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot. warnings.warn(msg, UserWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="seaborn.categorical")

In [ ]:
env_fname = ".dev-env" if training_mode else ".env"
assert dotenv.load_dotenv(f"{os.path.expanduser('~')}/work/{env_fname}", override=True)

In [ ]:
env = "dev" if training_mode else "prod"
engagement = "EC007"

if engagement is not None and engagement.lower().startswith("ec"):
    lake = procanswon.LakeV2(
        env=env,
        user=os.environ.get("USER"),
        ec_manager_base=f"https://ec-auth-manager.{env}.procan.org.au",
        engagement=engagement.lower(),
    )
    credentials = lake.get_pocket_credentials()
    username = credentials["username"]
    password = credentials["password"]
else:
    username = os.environ.get("TRINO_USERNAME")
    password = os.environ.get("TRINO_PASSWORD")

connect_args = {
    "http_scheme": "https",
    "user": username,
    "catalog": "iceberg",
    "schema": "pocket",
    "auth": trino.auth.BasicAuthentication(username, password),
}
SQL_ALCHEMY_ENGINE = create_engine(
    f"trino://trino.{env}.procan.org.au:443/",
    connect_args=connect_args,
)
SQL_ALCHEMY_CONNECTION = SQL_ALCHEMY_ENGINE.connect()

# This has to go after the engine is initialised
logging.getLogger("sqlalchemy").setLevel(logging.WARNING)

In [ ]:
def execute(query: str) -> pd.DataFrame:
    try:
        return pd.read_sql(text(query), SQL_ALCHEMY_CONNECTION)
    except Exception as e:
        logging.error(str(e))


# Conversion between skyline and pocket schemas

SKYLINE_COLUMNS = [
    "Replicate",
    "Protein",
    "Peptide",
    "Isotope Label Type",
    "Analyte Concentration",
    "RatioLightToHeavy",
    "Replicate Concentration",
    "Peptide Peak Found Ratio",
    "Peptide Retention Time",
    "DotProductLightToHeavy",
    "Normalized Area",
]

EXTRA_COLUMNS = [
    "Prep LIMS ID",
    "Dataset Name",
    "Notes",
    "Upload Time",
    "Excluded",
    "Engagement",
]

EXPECTED_DATASET_COLUMNS = SKYLINE_COLUMNS + EXTRA_COLUMNS


def unsnake_df(df: DataFrame):
    from_snake = {snake(r): r for r in EXPECTED_DATASET_COLUMNS}
    df.columns = [from_snake[c] for c in df.columns]


def snake(col: str) -> str:
    col = col.lower()
    col = col.replace(" ", "_").replace("-", "_").replace(".", "_")
    col = col.strip("_")
    return col


# Wrappers for different visualsation implementations between notebook vs web


def show_df(df: pd.DataFrame):
    if len(df) == 0:
        display("(Dataframe Empty)")
    else:
        rows, cols = df.shape
        display(f"Rows x Columns: {rows} x {cols}")
        display(df)


def show_plot(fig):
    pass


def show_error(message: str):
    logging.error(message)

## Info for Document Generation


In [ ]:
from dataclasses import dataclass
from matplotlib.figure import Figure
import tempfile


@dataclass
class Plot:
    path: str
    fig: Figure
    tighten: bool = False


@dataclass
class NotebookOutput:
    # General stats & plots
    patients: int
    samples: int
    reference_range_plot: Plot
    pan_cancer_percentile_plot: Plot

    # Patient Cancer specific stats and plots
    cancer_cat: str
    cancer_cat_patients: int
    cancer_cat_percentile_plot: Plot

    pan_cancer_concentration_plot: Plot
    pan_cancer_z_score_plot: Plot
    cancer_cat_z_score_plot: Plot

    reference_df: DataFrame


temp_dir = tempfile.TemporaryDirectory()

## Report Start


In [ ]:
if skyline_input_path:
    srm_data = pd.read_csv(skyline_input_path, sep="\t")
else:
    srm_data = execute("SELECT * FROM pocket.srm_data WHERE excluded = 'False'")
    unsnake_df(srm_data)

srm_data["Prep LIMS ID"] = srm_data["Replicate"].apply(lambda x: x.split("_")[2])
srm_prep_lims_ids = list(srm_data["Prep LIMS ID"].unique())

if len(srm_data) == 0:
    show_error("No SRM Data")
    srm_prep_lims_ids = []
else:
    prolims_data = execute(
        f""" SELECT prep_lims_id,
                        sample_cancer_cat_2 as Cancer_type,
                        sample_cancer_cat_1 as Cancer_Category,
                        sample_sample_descriptor as Tumour_Normal,
                        sample_sample_collection_site as Sample_Site
                FROM prolims_preps
                WHERE prep_lims_id IN {tuple(srm_prep_lims_ids)}"""
    )

    srm_data = srm_data.merge(
        prolims_data,
        how="left",
        left_on="Prep LIMS ID",
        right_on="prep_lims_id",
    ).drop(columns=["prep_lims_id"])

    srm_data.loc[srm_data["Tumour_Normal"] != "Malignant", "Tumour_Normal"] = "Normal"
    srm_data["Site_Tum_Nor"] = srm_data["Sample_Site"].map(str) + " - " + srm_data["Tumour_Normal"].map(str)
    srm_data["Protein"] = srm_data["Protein"].str.extract(r"\|([^|_]+)_HUMAN")
    show_df(srm_data)

## Select patient

### Type in the patient Prep_LIMS ID and click run analysis


In [ ]:
if prep_lims_id not in srm_prep_lims_ids:
    show_error(f"Prep LIMS ID {prep_lims_id} not found in srm data")

## ADC Targets

### List of the current ADC targets included in the report


In [ ]:
target_proteins = [
    "EGFR",
    "TACD2",
    "ERBB2",
    "CEAM5",
    "MET",
    "TF",
    "FOLR1",
    "CD19",
    "CD44",
    "CD33",
    "CD22",
    "PCNA",
    "UFO",
    "MUC18",
    "KI67",
    "GPC3",
    "CLD6",
    "CLD18",
    "CD276",
    "VTCN1",
    "NECT4",
]

In [ ]:
ADC_proteins = [
    "EGFR",
    "TACD2",
    "ERBB2",
    "CEAM5",
    "MET",
    "TF",
    "FOLR1",
    "CD19",
    "CD44",
    "CD33",
    "CD22",
    "UFO",
    "MUC18",
    "GPC3",
    "CLD6",
    "CLD18",
    "CD276",
    "VTCN1",
    "NECT4",
]

In [ ]:
Proliferative_proteins = ["PCNA", "KI67"]

In [ ]:
srm_data = srm_data[srm_data["Protein"].isin(target_proteins)]

In [ ]:
MS_File_Index = list(srm_data.Replicate)
MS_File_Index = pd.DataFrame(MS_File_Index)
MS_File_Index.columns = ["Replicate"]
# show_df(MS_File_Index)

In [ ]:
MS_File_Index2 = MS_File_Index
MS_File_Index2 = MS_File_Index2.Replicate.str.split("_", expand=True)
MS_File_Index2.columns = [
    "Date",
    "Method_LIMS",
    "Prep_LIMS",
    "MS_LIMS",
    "Sample",
    "Acquisition",
]
# show_df(MS_File_Index2)

In [ ]:
data2 = MS_File_Index.join(MS_File_Index2).merge(srm_data)
# show_df(data2)

In [ ]:
Average_peptide = (
    data2.groupby(
        [
            "Prep_LIMS",
            "Cancer_type",
            "Cancer_Category",
            "Tumour_Normal",
            "Sample_Site",
            "Site_Tum_Nor",
            "Protein",
            "Peptide",
        ]
    )["Replicate Concentration"]
    .agg(Peptide_average="mean", Peptide_StDev="std")
    .reset_index()
)

Average_peptide["Peptide_CV"] = Average_peptide["Peptide_StDev"] / Average_peptide["Peptide_average"]

show_df(Average_peptide)

In [ ]:
Average_peptide.shape

### Any files with high CV values between peptide replicates will be listed in the table below, requiring review and QC by the targeted protoemics team before proceeding


In [ ]:
high_peptide_CV = Average_peptide[Average_peptide["Peptide_CV"] > 5]
show_df(high_peptide_CV)

In [ ]:
Average_Protein = (
    Average_peptide.groupby(
        ["Prep_LIMS", "Cancer_type", "Cancer_Category", "Tumour_Normal", "Sample_Site", "Site_Tum_Nor", "Protein"]
    )["Peptide_average"]
    .agg(Protein_average="mean", Protein_StDev="std")
    .reset_index()
)
Average_Protein["Protein_CV"] = Average_Protein["Protein_StDev"] / Average_Protein["Protein_average"]

# show_df(Average_Protein)

### Any files with high CV for the protein absolute concentration will be listed in the table below, requiring review and QC by the targeted proteomics team before proceeding


In [ ]:
high_protein_CV = Average_Protein[Average_Protein["Protein_CV"] > 5]
show_df(high_protein_CV)

### Pan-cancer Normal and Maligant samples = Protein_percentile


In [ ]:
protein_stats = Average_Protein.groupby("Protein")["Protein_average"].agg(["mean", "std"]).reset_index()
protein_stats.columns = ["Protein", "Mean_Protein_Percentile_Avg", "Std_Protein_Percentile_Avg"]
Protein_percentile = Average_Protein.merge(protein_stats, on="Protein", how="left")
Protein_percentile["z_score"] = (
    Protein_percentile["Protein_average"] - Protein_percentile["Mean_Protein_Percentile_Avg"]
) / Protein_percentile["Std_Protein_Percentile_Avg"]
Protein_percentile["Protein_percentile"] = norm.cdf(Protein_percentile["z_score"])

In [ ]:
Protein_percentile["TargetQuant_panel"] = Protein_percentile["Protein"]

In [ ]:
Protein_percentile.loc[Protein_percentile["Protein"].isin(ADC_proteins), "TargetQuant_panel"] = "ADC"
Protein_percentile.loc[Protein_percentile["Protein"].isin(Proliferative_proteins), "TargetQuant_panel"] = (
    "Proliferative"
)

In [ ]:
show_df(Protein_percentile)

In [ ]:
Patient_Prep_LIMS = [prep_lims_id]

### Patients percentile relative to Pan-cancer normal and malignant sample = Patient_Data


In [ ]:
Patient_Data = Protein_percentile.loc[Protein_percentile["Prep_LIMS"].isin(Patient_Prep_LIMS)].copy()
Patient_Data.sort_values(by=["TargetQuant_panel", "Protein_percentile"], ascending=[True, False], inplace=True)
Patient_Data["Patient percentile"] = Patient_Data["Protein_percentile"] * 100
Patient_Data["Patient percentile"] = Patient_Data["Patient percentile"].apply(lambda x: int(round(x, 0)))
Patient_Data["Protein_percentile_axis"] = (
    Patient_Data["TargetQuant_panel"].map(str)
    + " - "
    + Patient_Data["Protein"].map(str)
    + " - "
    + Patient_Data["Patient percentile"].map(str)
)
show_df(Patient_Data)

In [ ]:
if len(Patient_Data) == 0:
    show_error(f"No Patient (i.e., ProLIMS) data for Prep LIMS ID {prep_lims_id}")

## Patient's Cancer Category and Type


In [ ]:
Cancer_Category = Patient_Data["Cancer_Category"].unique().item()
Cancer_Type = Patient_Data["Cancer_type"].unique().item()
print("Cancer category:", Cancer_Category, "and Cancer type:", Cancer_Type)

In [ ]:
Cancer_Type_label = Cancer_Category + " patients"
Cancer_Type2_label = Cancer_Type + " patients"
Cancer_Type_range = Cancer_Category + " range"
Cancer_Type_percentile = Cancer_Category + " percentile"
Cancer_Type_zscore = Cancer_Category + " z-score"

In [ ]:
Protein_names = Patient_Data.Protein.unique()
PanCancer_yaxis = Patient_Data.Protein_percentile_axis.unique()

### Cancer type as a percentile of Pan-cancer for normal and malignant samples = Cancer_type_data


In [ ]:
Cancer_type_data = Protein_percentile
Cancer_type_data = Cancer_type_data.loc[
    (Cancer_type_data["Cancer_Category"] == Cancer_Category) & (Cancer_type_data["Tumour_Normal"] == "Malignant")
]
# show_df(Cancer_type_data)

## Reference range

### The number of patient samples and cancer types that make up the pan-cancer reference range


In [ ]:
sns.set_style("white")

In [ ]:
Reference_count = (
    Protein_percentile.groupby(["Cancer_Category", "Tumour_Normal"])["Prep_LIMS"]
    .nunique()
    .reset_index()
    .rename(columns={"Prep_LIMS": "Unique_Prep_LIMS_Count"})
)

# show_df(Reference_count)

In [ ]:
Total_Cancer_references = Protein_percentile[Protein_percentile["Tumour_Normal"] == "Malignant"]
Total_Cancer_references = Total_Cancer_references["Prep_LIMS"].nunique()
Total_Normal_references = Protein_percentile[Protein_percentile["Tumour_Normal"] == "Normal"]
Total_Normal_references = Total_Normal_references["Prep_LIMS"].nunique()
Total_CancerType_references = Cancer_type_data.Prep_LIMS.nunique()

In [ ]:
print("Pan-Cancer reference:", Total_Cancer_references, "Patients")
print("Normal reference:", Total_Normal_references, "samples")
print(Cancer_Category, "reference:", Total_CancerType_references, "Patients")

### Cancer type data as a percentile of the cancer type normal and malignant = Protein_percentile2


In [ ]:
Cancer_type_data2 = data2
Cancer_type_data2 = Cancer_type_data2.loc[(Cancer_type_data2["Cancer_Category"] == Cancer_Category)]
# show_df(Cancer_type_data2)

In [ ]:
Average_peptide2 = (
    Cancer_type_data2.groupby(["Prep_LIMS", "Cancer_Category", "Tumour_Normal", "Sample_Site", "Protein", "Peptide"])[
        "Replicate Concentration"
    ]
    .agg(Peptide_average2="mean", Peptide_StDev2="std")
    .reset_index()
)

Average_peptide2["Peptide_CV"] = Average_peptide2["Peptide_StDev2"] / Average_peptide2["Peptide_average2"]
show_df(Average_peptide2)

In [ ]:
Average_Protein2 = (
    Average_peptide2.groupby(["Prep_LIMS", "Cancer_Category", "Tumour_Normal", "Sample_Site", "Protein"])[
        "Peptide_average2"
    ]
    .agg(Protein_average2="mean", Protein_StDev2="std")
    .reset_index()
)
Average_Protein2["Protein_CV"] = Average_Protein2["Protein_StDev2"] / Average_Protein2["Protein_average2"]
show_df(Average_Protein2)

In [ ]:
protein_stats2 = Average_Protein2.groupby("Protein")["Protein_average2"].agg(["mean", "std"]).reset_index()
protein_stats2.columns = ["Protein", "mean_protein_avg", "std_protein_avg"]
Protein_percentile2 = Average_Protein2.merge(protein_stats2, on="Protein", how="left")
Protein_percentile2["z_score"] = (
    Protein_percentile2["Protein_average2"] - Protein_percentile2["mean_protein_avg"]
) / Protein_percentile2["std_protein_avg"]
Protein_percentile2["Protein_percentile2"] = norm.cdf(Protein_percentile2["z_score"])
show_df(Protein_percentile2)

In [ ]:
Protein_percentile2.loc[Protein_percentile2["Protein"].isin(ADC_proteins), "TargetQuant_panel"] = "ADC"
Protein_percentile2.loc[Protein_percentile2["Protein"].isin(Proliferative_proteins), "TargetQuant_panel"] = (
    "Proliferative"
)
show_df(Protein_percentile2)

### df of patients cancer cat 2 only with normal and tumour


In [ ]:
Cancer_type_data3 = Protein_percentile
Cancer_type_data3 = Cancer_type_data3.loc[
    (Cancer_type_data3["Cancer_type"] == Cancer_Type) & (Cancer_type_data3["Tumour_Normal"] == "Malignant")
]
# show_df(Cancer_type_data3)

### Patient data as a percentile of the cancer type normal and malignant samples


In [ ]:
Patient_Data2 = Protein_percentile2.loc[Protein_percentile2["Prep_LIMS"].isin(Patient_Prep_LIMS)].copy()
Patient_Data2["Protein_ordered"] = pd.Categorical(Patient_Data2["Protein"], categories=Protein_names, ordered=True)
Patient_Data2 = Patient_Data2.sort_values(by="Protein_ordered")
Patient_Data2["Patient percentile"] = Patient_Data2["Protein_percentile2"] * 100
Patient_Data2["Patient percentile"] = Patient_Data2["Patient percentile"].apply(lambda x: int(round(x, 0)))
Patient_Data2["Protein_percentile_axis2"] = (
    Patient_Data2["TargetQuant_panel"].map(str)
    + " - "
    + Patient_Data2["Protein"].map(str)
    + " - "
    + Patient_Data2["Patient percentile"].map(str)
)
show_df(Patient_Data2)

In [ ]:
Protein_names2 = Patient_Data2.Protein.unique()
PanCancer_yaxis2 = Patient_Data2.Protein_percentile_axis2.unique()

### Pan-cancer df with malignant samples only = CancerOnly_Protein_percentile


In [ ]:
CancerOnly_Average_peptide = data2.loc[data2["Tumour_Normal"] == "Malignant"]

In [ ]:
CancerOnly_Average_peptide = (
    CancerOnly_Average_peptide.groupby(
        [
            "Prep_LIMS",
            "Cancer_type",
            "Cancer_Category",
            "Tumour_Normal",
            "Sample_Site",
            "Site_Tum_Nor",
            "Protein",
            "Peptide",
        ]
    )["Replicate Concentration"]
    .agg(Peptide_average="mean", Peptide_StDev="std")
    .reset_index()
)

CancerOnly_Average_peptide["Peptide_CV"] = (
    CancerOnly_Average_peptide["Peptide_StDev"] / CancerOnly_Average_peptide["Peptide_average"]
)

# show_df (CancerOnly_Average_peptide)

In [ ]:
CancerOnly_Average_peptide.shape

In [ ]:
CancerOnly_Average_Protein = (
    CancerOnly_Average_peptide.groupby(
        ["Prep_LIMS", "Cancer_type", "Cancer_Category", "Tumour_Normal", "Sample_Site", "Site_Tum_Nor", "Protein"]
    )["Peptide_average"]
    .agg(Protein_average="mean", Protein_StDev="std")
    .reset_index()
)
CancerOnly_Average_Protein["Protein_CV"] = (
    CancerOnly_Average_Protein["Protein_StDev"] / CancerOnly_Average_Protein["Protein_average"]
)

# show_df(CancerOnly_Average_Protein)

In [ ]:
CancerOnly_protein_stats = (
    CancerOnly_Average_Protein.groupby("Protein")["Protein_average"].agg(["mean", "std"]).reset_index()
)
CancerOnly_protein_stats.columns = ["Protein", "Mean_Protein_Percentile_Avg", "Std_Protein_Percentile_Avg"]
CancerOnly_Protein_percentile = CancerOnly_Average_Protein.merge(CancerOnly_protein_stats, on="Protein", how="left")
CancerOnly_Protein_percentile["z_score"] = (
    CancerOnly_Protein_percentile["Protein_average"] - CancerOnly_Protein_percentile["Mean_Protein_Percentile_Avg"]
) / CancerOnly_Protein_percentile["Std_Protein_Percentile_Avg"]
CancerOnly_Protein_percentile["Protein_percentile"] = norm.cdf(CancerOnly_Protein_percentile["z_score"])
# CancerOnly_Protein_percentile

### Patient percentile relative to only malignant samples = CancerOnly_Patient_Data


In [ ]:
CancerOnly_Patient_Data = CancerOnly_Protein_percentile.loc[
    CancerOnly_Protein_percentile["Prep_LIMS"].isin(Patient_Prep_LIMS)
].copy()
CancerOnly_Patient_Data.sort_values(by="Protein_percentile", ascending=False, inplace=True)
CancerOnly_Patient_Data["Patient percentile"] = CancerOnly_Patient_Data["Protein_percentile"] * 100
CancerOnly_Patient_Data["Patient percentile"] = CancerOnly_Patient_Data["Patient percentile"].apply(
    lambda x: int(round(x, 0))
)
CancerOnly_Patient_Data["Protein_percentile_axis"] = (
    CancerOnly_Patient_Data["Protein"].map(str)
    + " - "
    + CancerOnly_Patient_Data["Patient percentile"].map(str)
    + " percentile"
)
# show_df(CancerOnly_Patient_Data)

In [ ]:
Protein_names3 = CancerOnly_Patient_Data.Protein.unique()
PanCancer_yaxis3 = CancerOnly_Patient_Data.Protein.unique()

### Cancer type df as a percentile of cancer type malignant samples only = CancerOnly_Protein_percentile2


In [ ]:
CancerOnly_type_data2 = data2
CancerOnly_type_data2 = CancerOnly_type_data2.loc[(CancerOnly_type_data2["Cancer_Category"] == Cancer_Category)]
CancerOnly_type_data2 = CancerOnly_type_data2.loc[(CancerOnly_type_data2["Tumour_Normal"] == "Malignant")]
# show_df(CancerOnly_type_data2)

In [ ]:
CancerOnly_Average_peptide2 = (
    CancerOnly_type_data2.groupby(
        ["Prep_LIMS", "Cancer_Category", "Tumour_Normal", "Sample_Site", "Protein", "Peptide"]
    )["Replicate Concentration"]
    .agg(Peptide_average2="mean", Peptide_StDev2="std")
    .reset_index()
)

CancerOnly_Average_peptide2["Peptide_CV"] = (
    CancerOnly_Average_peptide2["Peptide_StDev2"] / CancerOnly_Average_peptide2["Peptide_average2"]
)
# show_df(CancerOnly_Average_peptide2)

In [ ]:
CancerOnly_Average_Protein2 = (
    CancerOnly_Average_peptide2.groupby(["Prep_LIMS", "Cancer_Category", "Tumour_Normal", "Sample_Site", "Protein"])[
        "Peptide_average2"
    ]
    .agg(Protein_average2="mean", Protein_StDev2="std")
    .reset_index()
)
CancerOnly_Average_Protein2["Protein_CV"] = (
    CancerOnly_Average_Protein2["Protein_StDev2"] / CancerOnly_Average_Protein2["Protein_average2"]
)
# show_df(CancerOnly_Average_Protein2)

In [ ]:
CancerOnly_protein_stats2 = (
    CancerOnly_Average_Protein2.groupby("Protein")["Protein_average2"].agg(["mean", "std"]).reset_index()
)
CancerOnly_protein_stats2.columns = ["Protein", "mean_protein_avg", "std_protein_avg"]
CancerOnly_Protein_percentile2 = CancerOnly_Average_Protein2.merge(protein_stats2, on="Protein", how="left")
CancerOnly_Protein_percentile2["z_score"] = (
    CancerOnly_Protein_percentile2["Protein_average2"] - CancerOnly_Protein_percentile2["mean_protein_avg"]
) / Protein_percentile2["std_protein_avg"]
CancerOnly_Protein_percentile2["Protein_percentile2"] = norm.cdf(CancerOnly_Protein_percentile2["z_score"])
# show_df(CancerOnly_Protein_percentile2)

### Patient percentile relative to only the cancer type and only malignant samples = CancerOnly_Patient_Data2


In [ ]:
CancerOnly_Patient_Data2 = CancerOnly_Protein_percentile2.loc[
    CancerOnly_Protein_percentile2["Prep_LIMS"].isin(Patient_Prep_LIMS)
].copy()
CancerOnly_Patient_Data2["Protein_ordered"] = pd.Categorical(
    CancerOnly_Patient_Data2["Protein"], categories=Protein_names3, ordered=True
)
CancerOnly_Patient_Data2 = CancerOnly_Patient_Data2.sort_values(by="Protein_ordered")
CancerOnly_Patient_Data2["Patient percentile"] = CancerOnly_Patient_Data2["Protein_percentile2"] * 100
CancerOnly_Patient_Data2["Patient percentile"] = CancerOnly_Patient_Data2["Patient percentile"].apply(
    lambda x: int(round(x, 0))
)
CancerOnly_Patient_Data2["Protein_percentile_axis2"] = (
    CancerOnly_Patient_Data2["Protein"].map(str)
    + " - "
    + CancerOnly_Patient_Data2["Patient percentile"].map(str)
    + " percentile"
)
# show_df(CancerOnly_Patient_Data2)

In [ ]:
# Protein_percentile = All cancer and BOTH tumour and normal
# Patient_Data = Patient data relative to All cancer and BOTH tumour and normal
# Cancer_type_data = Only SAME cancer type and BOTH tumour and normal
# Patient_Data2 = Patient data relative to SAME cancer type and both tumour and normal
# Cancer_type_data3 = Only SAME cancer cat 2 and BOTH tumour normal from pan-cancer analysis

# CancerOnly_Protein_percentile = All cancer and ONLY tumour
# CancerOnly_Patient_Data = Patient data relative to ALL cancers and ONLY tumours
# CancerOnly_Protein_percentile2 = Only SAME cancer and ONLY tumours
# CancerOnly_Patient_Data2 = Pateints data relative to Only the same cancer type and only malignant samples

## Reference range - include in report


In [ ]:
fig, axes = plt.subplots(figsize=(5, 10))
g = sns.barplot(
    ax=axes,
    data=Reference_count,
    y="Cancer_Category",
    x="Unique_Prep_LIMS_Count",
    palette="Blues_r",
    hue="Tumour_Normal",
    hue_order=("Malignant", "Normal"),
    linewidth=1.0,
    edgecolor="0.2",
    dodge=True,
    gap=2,
)
xticklabels = Reference_count.Cancer_Category.unique()
g.set_xlabel("Number of patient samples", fontsize=14)
for container in g.containers:
    g.bar_label(container, fontsize=12, label_type="edge", padding=2)
g.set_ylabel("", fontsize=14, labelpad=-1)
g.set_yticklabels(xticklabels, rotation=0, fontsize=12)

plt.savefig(f"{today}_TQ_ADC_panel_ReferenceRange.png", dpi=800, bbox_inches="tight")

reference_range_plot = Plot(
    path=f"{temp_dir}/reference_range_plot.png",
    fig=fig,
    tighten=True,
)
show_plot(fig)

## Pan-cancer percentile w cancer type - include in report


In [ ]:
fig, axes = plt.subplots(figsize=(7.5, 13))
g = sns.boxplot(
    ax=axes,
    data=Protein_percentile,
    x="Protein_percentile",
    color="lightgrey",
    saturation=0.5,
    y="Protein",
    order=Protein_names,
    linewidth=1.5,
    linecolor="black",
    showfliers=False,
    hue="Tumour_Normal",
    hue_order=["Normal", "Malignant"],
    palette=["white", "lightgrey"],
    fliersize=4,
)
g = sns.swarmplot(
    ax=axes,
    data=CancerOnly_Protein_percentile2,
    x="Protein_percentile2",
    y="Protein",
    palette="Blues",
    hue="Protein_percentile2",
    legend=True,
    size=5,
    edgecolor="black",
    linewidth=0.1,
    order=Protein_names,
)
g = sns.swarmplot(
    ax=axes,
    data=Patient_Data,
    x="Protein_percentile",
    color="darkred",
    legend=False,
    size=15,
    edgecolor="white",
    linewidth=3,
    y="Protein",
    order=Protein_names,
)
Patient_legend = mlines.Line2D(
    [],
    [],
    color="darkred",
    marker="o",
    markeredgewidth=3,
    markeredgecolor="white",
    markersize=15,
    label="Patient",
    dash_capstyle="round",
    linestyle="None",
)
Pancancer_legend = mlines.Line2D(
    [],
    [],
    color="lightgrey",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=15,
    label="Pan-cancer range",
    dash_capstyle="round",
    linestyle="None",
)
Normal_legend = mlines.Line2D(
    [],
    [],
    color="white",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=14,
    label="Normal tissue range",
    dash_capstyle="round",
    linestyle="None",
)
Cancertype_legend = mlines.Line2D(
    [],
    [],
    color="lightblue",
    marker="o",
    markeredgewidth=2,
    markeredgecolor="lightgrey",
    markersize=15,
    label=Cancer_Type_label,
    dash_capstyle="round",
    linestyle="None",
)
plt.legend(
    handles=[Patient_legend, Normal_legend, Pancancer_legend, Cancertype_legend],
    bbox_to_anchor=(1.02, 1.20),
    fontsize=12,
)
g.set_xlabel("Pan-cancer percentile", fontsize=18)
g.set_ylabel("Percentile (nth)", loc="top", rotation=0, fontsize=16, labelpad=-1)
g.yaxis.set_label_coords(-0.02, 1.008)
g.set_yticklabels(PanCancer_yaxis, fontsize=14, ha="right")
g.xaxis.set_label_coords(0.5, 1.07)
plt.tight_layout()

plt.savefig(f"{today}_TQ_ADC_panel_PanCancer_{prep_lims_id}.png", dpi=800, bbox_inches="tight")

pan_cancer_percentile_plot = Plot(
    path=f"{temp_dir}/pan_cancer_percentile_plot.png",
    fig=fig,
)
show_plot(fig)

## Pancer-cancer percentile with all cancers - exclude from report


In [ ]:
fig, axes = plt.subplots(figsize=(7.5, 13))
g = sns.boxplot(
    ax=axes,
    data=Protein_percentile,
    x="Protein_percentile",
    color="lightgrey",
    saturation=0.5,
    y="Protein",
    order=Protein_names,
    linewidth=1.5,
    linecolor="black",
    showfliers=False,
    hue="Tumour_Normal",
    hue_order=["Normal", "Malignant"],
    palette=["white", "lightgrey"],
)
g = sns.stripplot(
    ax=axes,
    data=CancerOnly_Protein_percentile,
    x="Protein_percentile",
    y="Protein",
    palette="Blues",
    hue="Protein_percentile",
    legend=True,
    size=5,
    edgecolor="black",
    linewidth=0.1,
    order=Protein_names,
)
g = sns.swarmplot(
    ax=axes,
    data=Patient_Data,
    x="Protein_percentile",
    color="darkred",
    legend=False,
    size=15,
    edgecolor="white",
    linewidth=3,
    y="Protein",
    order=Protein_names,
)
Patient_legend = mlines.Line2D(
    [],
    [],
    color="darkred",
    marker="o",
    markeredgewidth=3,
    markeredgecolor="white",
    markersize=15,
    label="Patient",
    dash_capstyle="round",
    linestyle="None",
)
Pancancer_legend = mlines.Line2D(
    [],
    [],
    color="lightgrey",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=15,
    label="Pan-cancer range",
    dash_capstyle="round",
    linestyle="None",
)
Normal_legend = mlines.Line2D(
    [],
    [],
    color="white",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=14,
    label="Normal tissue range",
    dash_capstyle="round",
    linestyle="None",
)
Cancertype_legend = mlines.Line2D(
    [],
    [],
    color="lightblue",
    marker="o",
    markeredgewidth=2,
    markeredgecolor="lightgrey",
    markersize=15,
    label="Pan-cancer malignant samples",
    dash_capstyle="round",
    linestyle="None",
)
plt.legend(
    handles=[Patient_legend, Normal_legend, Pancancer_legend, Cancertype_legend],
    bbox_to_anchor=(1.02, 1.20),
    fontsize=12,
)
g.set_xlabel("Pan-cancer percentile", fontsize=18)
g.set_ylabel("Percentile (nth)", loc="top", rotation=0, fontsize=16, labelpad=-1)
g.yaxis.set_label_coords(-0.02, 1.008)
g.set_yticklabels(PanCancer_yaxis2, fontsize=14)
g.xaxis.set_label_coords(0.6, 1.07)
plt.tight_layout()

# plt.savefig(f"{today}_TQ_ADC_panel_PanCancer_{prep_lims_id}.png", dpi=800, bbox_inches='tight')

## Cancer type percentile - include in report


In [ ]:
fig, axes = plt.subplots(figsize=(7.5, 13))
g = sns.boxplot(
    ax=axes,
    data=Protein_percentile2,
    x="Protein_percentile2",
    color="lightgrey",
    saturation=0.5,
    y="Protein",
    order=Protein_names2,
    linewidth=1.5,
    linecolor="black",
    showfliers=False,
    hue="Tumour_Normal",
    hue_order=["Normal", "Malignant"],
    palette=["white", "lightgrey"],
)
g = sns.swarmplot(
    ax=axes,
    data=CancerOnly_Protein_percentile2,
    x="Protein_percentile2",
    y="Protein",
    palette="Blues",
    hue="Protein_percentile2",
    legend=True,
    size=5,
    edgecolor="black",
    linewidth=0.1,
    order=Protein_names2,
)
g = sns.swarmplot(
    ax=axes,
    data=Patient_Data2,
    x="Protein_percentile2",
    color="darkred",
    legend=False,
    size=15,
    edgecolor="white",
    linewidth=3,
    y="Protein",
    order=Protein_names2,
)
Patient_legend = mlines.Line2D(
    [],
    [],
    color="darkred",
    marker="o",
    markeredgewidth=3,
    markeredgecolor="white",
    markersize=15,
    label="Patient",
    dash_capstyle="round",
    linestyle="None",
)
Pancancer_legend = mlines.Line2D(
    [],
    [],
    color="lightgrey",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=15,
    label=Cancer_Type_range,
    dash_capstyle="round",
    linestyle="None",
)
Normal_legend = mlines.Line2D(
    [],
    [],
    color="white",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=14,
    label="Normal tissue range",
    dash_capstyle="round",
    linestyle="None",
)
Cancertype_legend = mlines.Line2D(
    [],
    [],
    color="lightblue",
    marker="o",
    markeredgewidth=2,
    markeredgecolor="lightgrey",
    markersize=15,
    label=Cancer_Type_label,
    dash_capstyle="round",
    linestyle="None",
)
plt.legend(
    handles=[Patient_legend, Normal_legend, Pancancer_legend, Cancertype_legend],
    bbox_to_anchor=(1.02, 1.20),
    fontsize=12,
)
g.set_xlabel("\n".join(wrap(Cancer_Type_percentile, width=30)), fontsize=18)
g.set_ylabel("Percentile (nth)", loc="top", rotation=0, fontsize=16, labelpad=-1)
g.yaxis.set_label_coords(-0.02, 1.008)
g.set_yticklabels(PanCancer_yaxis2, fontsize=14)
g.xaxis.set_label_coords(0.6, 1.07)
plt.tight_layout()

cancer_cat_percentile_plot = Plot(
    path=f"{temp_dir}/cancer_cat_percentile_plot.png",
    fig=fig,
)
show_plot(fig)

## Pan-cancer abundance with all patients - exclude from report


In [ ]:
fig, axes = plt.subplots(figsize=(7.5, 13))
g = sns.boxplot(
    ax=axes,
    data=Protein_percentile,
    x="Protein_average",
    color="lightgrey",
    saturation=0.5,
    y="Protein",
    order=Protein_names,
    linewidth=1.5,
    linecolor="black",
    fliersize=4,
    hue="Tumour_Normal",
    hue_order=["Normal", "Malignant"],
    palette=["white", "lightgrey"],
    log_scale=True,
)
g = sns.stripplot(
    ax=axes,
    data=Protein_percentile,
    x="Protein_average",
    y="Protein",
    palette="Purples",
    hue="Protein_percentile",
    legend=True,
    size=5,
    edgecolor="black",
    linewidth=0.2,
    order=Protein_names,
    log_scale=True,
    jitter=0.1,
)
g = sns.swarmplot(
    ax=axes,
    data=Patient_Data,
    x="Protein_average",
    color="darkred",
    legend=False,
    size=15,
    edgecolor="white",
    linewidth=3,
    y="Protein",
    order=Protein_names,
    log_scale=True,
)
Patient_legend = mlines.Line2D(
    [],
    [],
    color="darkred",
    marker="o",
    markeredgewidth=3,
    markeredgecolor="white",
    markersize=15,
    label="Patient",
    dash_capstyle="round",
    linestyle="None",
)
Pancancer_legend = mlines.Line2D(
    [],
    [],
    color="lightgrey",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=15,
    label="Pan-cancer range",
    dash_capstyle="round",
    linestyle="None",
)
Normal_legend = mlines.Line2D(
    [],
    [],
    color="white",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=14,
    label="Normal tissue range",
    dash_capstyle="round",
    linestyle="None",
)
Cancertype_legend = mlines.Line2D(
    [],
    [],
    color="purple",
    marker="o",
    markeredgewidth=2,
    markeredgecolor="lightgrey",
    markersize=15,
    label="Pan-cancer patients",
    dash_capstyle="round",
    linestyle="None",
)
plt.legend(
    handles=[Patient_legend, Normal_legend, Pancancer_legend, Cancertype_legend],
    bbox_to_anchor=(1.02, 1.12),
    fontsize=12,
)
g.set_xlabel("Pan-cancer absolute concentration (log10 fmol)", fontsize=16)
g.set_ylabel("Target", loc="top", rotation=0, fontsize=14, labelpad=-1)
g.yaxis.set_label_coords(-0.07, 1.005)
g.set_yticklabels(PanCancer_yaxis, fontsize=14)
g.xaxis.set_label_coords(0.5, 1.02)
plt.tight_layout()
# plt.savefig(f"{today}_TQ_ADC_panel_PanCancer_Abundances_{prep_lims_id}.png", dpi=800, bbox_inches='tight')

## Pan-Cancer abundances with cancer type patients - include in report


In [ ]:
fig, axes = plt.subplots(figsize=(10, 14))
g = sns.boxplot(
    ax=axes,
    data=Protein_percentile,
    x="Protein_average",
    color="lightgrey",
    saturation=0.5,
    y="Protein",
    order=Protein_names,
    linewidth=1.5,
    linecolor="black",
    showfliers=False,
    hue="Tumour_Normal",
    hue_order=["Normal", "Malignant"],
    palette=["white", "lightgrey"],
    log_scale=10,
)
g = sns.swarmplot(
    ax=axes,
    data=CancerOnly_Protein_percentile2,
    x="Protein_average2",
    y="Protein",
    palette="Purples",
    hue="Protein_percentile2",
    legend=True,
    size=5,
    edgecolor="black",
    linewidth=0.1,
    order=Protein_names,
    log_scale=10,
)
g = sns.stripplot(
    ax=axes,
    data=Cancer_type_data3,
    x="Protein_average",
    y="Protein",
    color="salmon",
    legend=True,
    size=5,
    edgecolor="black",
    linewidth=0.1,
    order=Protein_names,
    log_scale=10,
)
g = sns.swarmplot(
    ax=axes,
    data=Patient_Data,
    x="Protein_average",
    color="darkred",
    legend=False,
    size=15,
    edgecolor="white",
    linewidth=3,
    y="Protein",
    order=Protein_names,
    log_scale=10,
)
Patient_legend = mlines.Line2D(
    [],
    [],
    color="darkred",
    marker="o",
    markeredgewidth=3,
    markeredgecolor="white",
    markersize=15,
    label="Patient",
    dash_capstyle="round",
    linestyle="None",
)
Pancancer_legend = mlines.Line2D(
    [],
    [],
    color="lightgrey",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=15,
    label="Pan-cancer range",
    dash_capstyle="round",
    linestyle="None",
)
Normal_legend = mlines.Line2D(
    [],
    [],
    color="white",
    marker="s",
    markeredgewidth=1,
    markeredgecolor="Black",
    markersize=14,
    label="Normal tissue range",
    dash_capstyle="round",
    linestyle="None",
)
Cancertype_legend = mlines.Line2D(
    [],
    [],
    color="purple",
    marker="o",
    markeredgewidth=2,
    markeredgecolor="lightgrey",
    markersize=15,
    label=Cancer_Type_label,
    dash_capstyle="round",
    linestyle="None",
)
Cancertype_legend2 = mlines.Line2D(
    [],
    [],
    color="salmon",
    marker="o",
    markeredgewidth=2,
    markeredgecolor="lightgrey",
    markersize=15,
    label=Cancer_Type2_label,
    dash_capstyle="round",
    linestyle="None",
)
axes.legend(
    handles=[Patient_legend, Normal_legend, Pancancer_legend, Cancertype_legend, Cancertype_legend2],
    loc="upper left",
    bbox_to_anchor=(1.02, 1),
    borderaxespad=0.0,
    fontsize=12,
    frameon=False,
)

g.set_xlabel("Cancer type absolute concentration (log10 fmol)", fontsize=18)
g.set_ylabel("Target - nth", loc="top", rotation=0, fontsize=16, labelpad=-1)
g.yaxis.set_label_coords(-0.02, 1.005)
g.set_yticklabels(PanCancer_yaxis, fontsize=14)
g.xaxis.set_label_coords(0.5, 1.05)
plt.tight_layout()
plt.savefig(f"{today}_TQ_ADC_panel_PanCancer_wCancerType_Abundances_{prep_lims_id}.png", bbox_inches="tight")

pan_cancer_concentration_plot = Plot(
    path=f"{temp_dir}/pan_cancer_concentration_plot.png",
    fig=fig,
    tighten=True
)
show_plot(fig)

## Pan-cancer z-score - include in report


In [ ]:
vmin = min(CancerOnly_Patient_Data["z_score"].min(), -0.1)
vmax = max(CancerOnly_Patient_Data["z_score"].max(), 0.1)
two_slope_norm = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)
fig, axes = plt.subplots(figsize=(5, 10))
g = sns.barplot(
    ax=axes, data=CancerOnly_Patient_Data, y="Protein", x="z_score", linewidth=1.0, edgecolor="0.2", legend=False
)

cmap = sns.color_palette("vlag", as_cmap=True)
colors = cmap(two_slope_norm(CancerOnly_Patient_Data["z_score"]))
for bar, color in zip(g.patches, colors):
    bar.set_facecolor(color)

g.set_xlabel("Pan-cancer z-score", fontsize=14)
g.set_ylabel("", fontsize=14, labelpad=-1)
g.set_yticks(range(len(Protein_names)))
g.set_yticklabels(Protein_names3, rotation=0, fontsize=12)
g.xaxis.set_label_coords(0.5, 1.03)
g.axvline(0, color="black", linestyle="--")
plt.savefig(f"{today}_TQ_ADC_panel_PanCancerOnly_Zscore_{prep_lims_id}.png", dpi=800, bbox_inches="tight")

pan_cancer_z_score_plot = Plot(
    path=f"{temp_dir}/pan_cancer_z_score_plot.png",
    fig=fig,
    tighten=True,
)
show_plot(fig)

## Cancer type z-score - include in report


In [ ]:
vmin = min(CancerOnly_Patient_Data2["z_score"].min(), -0.1)
vmax = max(CancerOnly_Patient_Data2["z_score"].max(), 0.1)
two_slope_norm_2 = TwoSlopeNorm(vmin=vmin, vcenter=0, vmax=vmax)

fig, axes = plt.subplots(figsize=(5, 10))
g = sns.barplot(
    ax=axes,
    data=CancerOnly_Patient_Data2,
    y="Protein",
    x="z_score",
    linewidth=1.0,
    edgecolor="0.2",
    legend=False,
    order=Protein_names3,
)

cmap = sns.color_palette("vlag", as_cmap=True)
colors = cmap(two_slope_norm_2(CancerOnly_Patient_Data2["z_score"]))
for bar, color in zip(g.patches, colors):
    bar.set_facecolor(color)

g.set_xlabel(Cancer_Type_zscore, fontsize=14)
g.set_ylabel("", fontsize=14, labelpad=-1)
g.set_yticks(range(len(Protein_names)))
g.set_yticklabels(Protein_names3, rotation=0, fontsize=12)
g.xaxis.set_label_coords(0.5, 1.03)
g.axvline(0, color="black", linestyle="--")

plt.savefig(f"{today}_TQ_ADC_panel_CancerTypeOnly_Zscore_{prep_lims_id}.png", dpi=800, bbox_inches="tight")

cancer_cat_z_score_plot = Plot(
    path=f"{temp_dir}/cancer_cat_z_score_plot.png",
    fig=fig,
    tighten=True,
)
show_plot(fig)